In [7]:
import os
import re
import numpy as np
import pandas as pd
import xarray as xr
import arviz as az
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds
from numpy.linalg import solve, lstsq

# Limit threads
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# Input files (adjust paths if needed)
f_data  = "flow_data.csv"
f_score = "flow_data_score.csv"

# Output prefix/folder
out_prefix = "full_space_res"
out_dir = out_prefix

# -------------------------
# Helpers
# -------------------------
def normalize_name(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"[^\w_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def build_mass_balance(df_raw: pd.DataFrame, var_idx_map: pd.DataFrame, n_vars: int):
    """Build A_bal x = 0 for internal nodes (appear in both from_node_number and to_node_number)."""
    # Identify internal nodes in the full file (ignoring NaNs)
    from_all = df_raw["from_node_number"].dropna().astype(int).to_numpy() if "from_node_number" in df_raw.columns else np.array([], int)
    to_all   = df_raw["to_node_number"].dropna().astype(int).to_numpy()   if "to_node_number" in df_raw.columns   else np.array([], int)
    internal_nodes = set(from_all).intersection(set(to_all))

    # Restrict to our subset with assigned var_idx
    df_sub = var_idx_map.copy()
    df_sub["from_node_number"] = pd.to_numeric(df_sub.get("from_node_number", np.nan), errors="coerce")
    df_sub["to_node_number"]   = pd.to_numeric(df_sub.get("to_node_number",   np.nan), errors="coerce")

    A_rows = []
    for node in sorted(internal_nodes):
        inflow_vars  = df_sub.loc[df_sub["to_node_number"]   == node, "var_idx"].to_numpy(dtype=int)
        outflow_vars = df_sub.loc[df_sub["from_node_number"] == node, "var_idx"].to_numpy(dtype=int)
        if inflow_vars.size > 0 and outflow_vars.size > 0:
            row = np.zeros(n_vars, dtype=float)
            for j in inflow_vars:  row[j] += 1.0
            for j in outflow_vars: row[j] -= 1.0
            A_rows.append(row)

    if A_rows:
        A_bal = np.vstack(A_rows)
        b_bal = np.zeros(A_bal.shape[0], dtype=float)
    else:
        A_bal = np.zeros((0, n_vars), dtype=float)
        b_bal = np.zeros(0, dtype=float)
    return A_bal, b_bal

def nullspace(A, rtol=1e-10):
    if A.size == 0:
        return np.eye(A.shape[1])
    U, s, Vt = np.linalg.svd(A, full_matrices=True)
    rank = (s > rtol * s.max()).sum()
    Z = Vt[rank:].T
    return Z

# -------------------------
# 1) Load and preprocess data
# -------------------------
df_data = pd.read_csv(f_data, dtype=str)
df_score = pd.read_csv(f_score, dtype=str)

# Normalize names
for c in ["from_node_name", "to_node_name"]:
    if c in df_data.columns:
        df_data[c] = df_data[c].apply(normalize_name)
    if c in df_score.columns:
        df_score[c] = df_score[c].apply(normalize_name)

# Numeric columns
df_data["Value1"] = pd.to_numeric(df_data.get("Value1", np.nan), errors="coerce")
if "Flow index" in df_data.columns:
    df_data["Flow index"] = pd.to_numeric(df_data["Flow index"], errors="coerce")
if "from_node_number" in df_data.columns:
    df_data["from_node_number"] = pd.to_numeric(df_data["from_node_number"], errors="coerce")
if "to_node_number" in df_data.columns:
    df_data["to_node_number"] = pd.to_numeric(df_data["to_node_number"], errors="coerce")

# Merge scores onto data by normalized names; if Flow index exists in score, prefer that
join_cols = []
if "Flow index" in df_data.columns and "Flow index" in df_score.columns:
    df_score["Flow index"] = pd.to_numeric(df_score["Flow index"], errors="coerce")
    join_cols = ["Flow index"]
else:
    # Fallback to name-based join
    join_cols = [c for c in ["from_node_name", "to_node_name"] if c in df_data.columns and c in df_score.columns]

if join_cols:
    df = df_data.merge(df_score, on=join_cols, how="left", suffixes=("", "_score"))
else:
    df = df_data.copy()

# Observational uncertainty from score file: try common column names
score_std_col = next((c for c in ["std", "sigma", "noise_std", "obs_std", "Score", "score", "stddev"] if c in df.columns), None)
if score_std_col is not None:
    df["obs_std"] = pd.to_numeric(df[score_std_col], errors="coerce")
else:
    df["obs_std"] = np.nan

# Keep rows with both ends named
df = df.loc[(df.get("from_node_name", "") != "") & (df.get("to_node_name", "") != "")].copy()

# Assign variable indices:
# Prefer Flow index if present; else assign by order of unique (from,to) pairs
if "Flow index" in df.columns and df["Flow index"].notna().any():
    zero_based = int(df["Flow index"].dropna().min()) == 0
    df["var_idx"] = df["Flow index"].apply(lambda v: int(v) - (0 if zero_based else 1) if pd.notna(v) else np.nan)
else:
    pairs = df[["from_node_name", "to_node_name"]].drop_duplicates().reset_index(drop=True)
    pairs["var_idx"] = np.arange(len(pairs))
    df = df.merge(pairs, on=["from_node_name", "to_node_name"], how="left", suffixes=("", "_pair"))

# Drop invalid indices
df = df.loc[df["var_idx"].notna()].copy()
df["var_idx"] = df["var_idx"].astype(int)

# Aggregate duplicates per var_idx: median Value1, first nodes/numbers, std from score median
var_map = (
    df.groupby("var_idx", as_index=False)
      .agg(Value1=("Value1", "median"),
           from_node_name=("from_node_name", "first"),
           to_node_name=("to_node_name", "first"),
           from_node_number=("from_node_number", "first"),
           to_node_number=("to_node_number", "first"),
           obs_std=("obs_std", "median"))
)
n_vars = var_map.shape[0]
print(f"Variables detected: {n_vars}")

# Observations and uncertainties
y_obs = var_map["Value1"].to_numpy()
obs_mask = ~np.isnan(y_obs)

# Bounds: 20–180% of Value1 if observed, else wide bounds
max_obs = np.nanmax(y_obs) if np.any(obs_mask) else 1.0
sup_bound = float(max(1.0, max_obs * 1.8))
x_lb = np.where(obs_mask, 0.2 * y_obs, 0.0)
x_ub = np.where(obs_mask, 1.8 * y_obs, sup_bound)
x_lb = np.minimum(x_lb, x_ub - 1e-6)

# Observational std: use score if provided; else 10% heuristic for observed, large for missing
sigma_obs = var_map["obs_std"].to_numpy()
sigma_obs = np.where(np.isnan(sigma_obs), np.where(obs_mask, np.maximum(0.60 * np.abs(y_obs), 1e-3), np.inf), sigma_obs)
inv_obs_var = np.where(np.isfinite(sigma_obs), 1.0 / np.maximum(sigma_obs**2, 1e-12), 0.0)

# -------------------------
# 2) Mass-balance constraints for internal nodes (appear in both columns)
# -------------------------
A_bal, b_bal = build_mass_balance(df_data, var_map, n_vars)
print(f"Mass-balance constraints: {A_bal.shape[0]}")

# -------------------------
# 3) MAP optimization (quadratic) with bounds and equality constraints
# Objective: minimize sum_j (x_j - y_j)^2 / sigma_j^2
# -------------------------
Q = np.diag(inv_obs_var)
q = -(inv_obs_var * np.nan_to_num(y_obs, nan=0.0))

def obj_fun(x):
    return 0.5 * float(x @ (Q @ x)) + float(q @ x)

def obj_grad(x):
    return Q @ x + q

# Initial x0: use observations (or zeros), projected to mass balance and clipped
x0 = np.nan_to_num(y_obs, nan=0.0)
if A_bal.shape[0] > 0:
    rhs = A_bal @ x0
    AAT = A_bal @ A_bal.T
    try:
        lam = solve(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs)
    except np.linalg.LinAlgError:
        lam = lstsq(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs, rcond=None)[0]
    x0 = x0 - A_bal.T @ lam
x0 = np.clip(x0, x_lb, x_ub)

lin_con = LinearConstraint(A_bal, lb=b_bal, ub=b_bal) if A_bal.shape[0] > 0 else None
bounds = Bounds(x_lb, x_ub)

res = minimize(
    fun=obj_fun,
    x0=x0,
    jac=obj_grad,
    method="trust-constr",
    bounds=bounds,
    constraints=([lin_con] if lin_con is not None else []),
    options={"maxiter": 20000, "verbose": 1}
)

if not res.success:
    print("Warning: optimizer did not fully converge:", res.message)

x_map = res.x

# -------------------------
# 4) Posterior sampling via Laplace approximation on mass-balance nullspace
# -------------------------
def project_to_mass_balance(x):
    if A_bal.shape[0] == 0:
        return x
    rhs = A_bal @ x
    AAT = A_bal @ A_bal.T
    try:
        lam = solve(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs)
    except np.linalg.LinAlgError:
        lam = lstsq(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs, rcond=None)[0]
    return x - A_bal.T @ lam

Z = nullspace(A_bal)
Q_eff = Z.T @ Q @ Z
# Regularize and factor
try:
    L_eff = np.linalg.cholesky(Q_eff + 1e-12 * np.eye(Q_eff.shape[0]))
    chol_like = True
except np.linalg.LinAlgError:
    w, V = np.linalg.eigh(Q_eff)
    w = np.maximum(w, 1e-12)
    L_eff = V @ np.diag(np.sqrt(w))
    chol_like = False

n_chains, n_draws = 4, 2000
rng = np.random.default_rng(42)
samples_arr = np.empty((n_chains, n_draws, n_vars), dtype=float)
for ci in range(n_chains):
    for di in range(n_draws):
        z0 = rng.standard_normal(Q_eff.shape[0])
        if chol_like:
            # Solve L_eff g = z0
            g = solve(L_eff, z0)
        else:
            g = lstsq(L_eff, z0, rcond=None)[0]
        x_s = x_map + Z @ g
        x_s = np.clip(x_s, x_lb, x_ub)
        x_s = project_to_mass_balance(x_s)
        samples_arr[ci, di, :] = x_s

# -------------------------
# 5) Pack results and save
# -------------------------
coords = {"chain": np.arange(n_chains), "draw": np.arange(n_draws), "mu_x_dim_0": np.arange(n_vars)}
posterior_ds = xr.Dataset({"mu_x": (("chain", "draw", "mu_x_dim_0"), samples_arr)}, coords=coords)
idata = az.InferenceData(posterior=posterior_ds)

summary = az.summary(idata, var_names=["mu_x"])
print(summary)

x_posterior = summary["mean"].to_numpy()
assert x_posterior.shape[0] == n_vars

if not os.path.exists(out_dir):
    os.mkdir(out_dir)
else:
    print(f"The directory {out_dir} already exists.")

# Save samples and posterior
az.to_netcdf(idata, f"{out_prefix}.nc")
mu_flat = idata.posterior["mu_x"].stack(sample=("chain", "draw")).transpose("mu_x_dim_0", "sample").values
np.save(f"{out_prefix}.npy", mu_flat)

# Save posterior mean CSV (with metadata)
out_csv = os.path.join(out_dir, f"{out_prefix}_posterior_mean.csv")
pm_df = var_map.copy()
pm_df["posterior_mean"] = x_posterior
pm_df["map_estimate"]   = x_map
pm_df["lower_bound"]    = x_lb
pm_df["upper_bound"]    = x_ub
pm_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# Residuals (mass balance)
if A_bal.shape[0] > 0:
    mb_resid = A_bal @ x_posterior
    print("Mass-balance residuals (posterior mean): min", float(mb_resid.min()), "max", float(mb_resid.max()))

# Histograms
for j in range(n_vars):
    vals = mu_flat[j, :]
    plt.figure(figsize=(7, 4))
    plt.hist(vals, bins=50, density=True, alpha=0.6, color="steelblue")
    mean = vals.mean()
    low, high = np.quantile(vals, [0.025, 0.975])
    plt.axvline(mean, color="k", linestyle="--", label=f"mean = {mean:.3f}")
    plt.axvline(low,  color="red", linestyle=":", label=f"2.5% = {low:.3f}")
    plt.axvline(high, color="red", linestyle=":")
    plt.title(f"Posterior of μ — var {j}")
    plt.xlabel("μ")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"hist_{j}.png"))
    plt.close()

Variables detected: 241
Mass-balance constraints: 62
`xtol` termination condition is satisfied.
Number of iterations: 4826, function evaluations: 4908, CG iterations: 548312, optimality: 2.34e-04, constraint violation: 7.96e-13, execution time: 3.9e+02 s.
               mean        sd    hdi_3%   hdi_97%  mcse_mean  mcse_sd  \
mu_x[0]     241.273  1003.244 -1302.891  1767.789     11.095    5.394   
mu_x[1]    4330.135  2816.487   901.718  7806.502     31.192    6.578   
mu_x[2]      17.801   937.413 -1377.319  1419.642     10.373    5.063   
mu_x[3]     218.884   971.549 -1296.382  1744.035     10.720    5.338   
mu_x[4]    4334.722  2448.695  1085.213  7690.210     26.975    7.432   
...             ...       ...       ...       ...        ...      ...   
mu_x[236]  2523.666  2272.879    85.106  5193.204     25.425    3.186   
mu_x[237]  2523.977  2272.903    85.106  5193.204     25.425    3.185   
mu_x[238]  2524.779  2272.967    85.106  5193.204     25.426    3.181   
mu_x[239]  252

C:\Users\yongxian.zhu\AppData\Local\Temp\ipykernel_23924\605596014.py:305: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import arviz as az

# Paths (adjust if needed)
f_data = "flow_data.csv"
posterior_csv = os.path.join("full_space_res", "full_space_res_posterior_mean.csv")
posterior_nc = "full_space_res.nc"

# Thresholds for flagging large deviations
REL_THRESH = 1.0   # >100% relative deviation
ABS_THRESH = 0.1   # absolute deviation threshold (units of your flows)

def normalize_name(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"[^\w_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

# 1) Load and aggregate flow_data mean by variable index
df_data = pd.read_csv(f_data, dtype=str)
# Normalize names if present (useful for fallback name-based join)
for c in ["from_node_name", "to_node_name"]:
    if c in df_data.columns:
        df_data[c] = df_data[c].apply(normalize_name)

# Numeric conversions
df_data["Value1"] = pd.to_numeric(df_data.get("Value1", np.nan), errors="coerce")
if "Flow index" in df_data.columns:
    df_data["Flow index"] = pd.to_numeric(df_data["Flow index"], errors="coerce")

# Assign var_idx: prefer Flow index if available, else assign by unique (from,to) pairs
if "Flow index" in df_data.columns and df_data["Flow index"].notna().any():
    zero_based = int(df_data["Flow index"].dropna().min()) == 0
    df_data["var_idx"] = df_data["Flow index"].apply(lambda v: int(v) - (0 if zero_based else 1) if pd.notna(v) else np.nan)
else:
    if ("from_node_name" in df_data.columns) and ("to_node_name" in df_data.columns):
        pairs = df_data[["from_node_name", "to_node_name"]].drop_duplicates().reset_index(drop=True)
        pairs["var_idx"] = np.arange(len(pairs))
        df_data = df_data.merge(pairs, on=["from_node_name", "to_node_name"], how="left", suffixes=("", "_pair"))
    else:
        raise ValueError("Cannot assign var_idx: need 'Flow index' or both 'from_node_name' and 'to_node_name' in flow_data.csv")

df_data = df_data.loc[df_data["var_idx"].notna()].copy()
df_data["var_idx"] = df_data["var_idx"].astype(int)

# Aggregate duplicates per var_idx using mean of Value1 (your request: flow_data mean)
flow_mean = (
    df_data.groupby("var_idx", as_index=False)
           .agg(flow_data_mean=("Value1", "mean"),
                dup_count=("Value1", "size"),
                from_node_name=("from_node_name", "first") if "from_node_name" in df_data.columns else ("Value1", "size"),
                to_node_name=("to_node_name", "first")     if "to_node_name" in df_data.columns     else ("Value1", "size"))
)

# 2) Load posterior mean
if os.path.exists(posterior_csv):
    post_df = pd.read_csv(posterior_csv)
    # Expect var_idx column; if missing, create 0..N-1
    if "var_idx" not in post_df.columns:
        post_df["var_idx"] = np.arange(post_df.shape[0])
    post_df = post_df[["var_idx", "posterior_mean"] + ([c for c in ["from_node_name", "to_node_name"] if c in post_df.columns])]
else:
    # Fallback: derive from NetCDF produced earlier
    idata = az.from_netcdf(posterior_nc)
    post_mean = az.summary(idata, var_names=["mu_x"])["mean"].to_numpy()
    post_df = pd.DataFrame({"var_idx": np.arange(len(post_mean)), "posterior_mean": post_mean})

# 3) Align and compare
# Primary alignment: by var_idx
cmp = post_df.merge(flow_mean, on="var_idx", how="left", suffixes=("_post", "_flow"))

# Fallback: name-based alignment for any missing flow_data_mean (if names exist)
if ("flow_data_mean" in cmp.columns) and cmp["flow_data_mean"].isna().any():
    # build name key in both tables (normalized earlier)
    if ("from_node_name" in cmp.columns) and ("to_node_name" in cmp.columns) and ("from_node_name" in flow_mean.columns) and ("to_node_name" in flow_mean.columns):
        cmp["name_key"] = cmp["from_node_name"].fillna("") + "->" + cmp["to_node_name"].fillna("")
        flow_mean["name_key"] = flow_mean["from_node_name"].fillna("") + "->" + flow_mean["to_node_name"].fillna("")
        name_map = flow_mean[["name_key", "flow_data_mean"]].drop_duplicates()
        # fill missing by name_key
        mask_missing = cmp["flow_data_mean"].isna() & (cmp["name_key"] != "")
        cmp.loc[mask_missing, "flow_data_mean"] = cmp.loc[mask_missing, "name_key"].map(dict(zip(name_map["name_key"], name_map["flow_data_mean"])))

# Compute deviations for rows where flow_data_mean is available
cmp["abs_diff"] = cmp["posterior_mean"] - cmp["flow_data_mean"]
cmp["rel_diff"] = cmp["abs_diff"] / np.where(np.abs(cmp["flow_data_mean"]) > 1e-12, np.abs(cmp["flow_data_mean"]), 1.0)
cmp["flag_large_rel"] = np.abs(cmp["rel_diff"]) > REL_THRESH
cmp["flag_large_abs"] = np.abs(cmp["abs_diff"]) > ABS_THRESH

# Summary
available = cmp["flow_data_mean"].notna().sum()
total = cmp.shape[0]
pct = lambda m: 100.0 * float(np.sum(m)) / max(1, available)
print(f"Compared posterior vs flow_data mean on {available}/{total} variables.")
print(f"- mean |abs_diff|: {np.nanmean(np.abs(cmp['abs_diff'])):.4g}")
print(f"- median |abs_diff|: {np.nanmedian(np.abs(cmp['abs_diff'])):.4g}")
print(f"- mean |rel_diff|: {np.nanmean(np.abs(cmp['rel_diff'])):.4g}")
print(f"- >{int(REL_THRESH*100)}% relative deviation: {pct(np.abs(cmp['rel_diff']) > REL_THRESH):.1f}%")
print(f"- >{ABS_THRESH} absolute deviation: {pct(np.abs(cmp['abs_diff']) > ABS_THRESH):.1f}%")

# Top deviations by relative difference
cmp_sorted = cmp.sort_values(by="rel_diff", key=lambda s: np.abs(s), ascending=False)
topN = 20
cols_show = ["var_idx", "posterior_mean", "flow_data_mean", "abs_diff", "rel_diff", "dup_count"]
if "from_node_name" in cmp_sorted.columns and "to_node_name" in cmp_sorted.columns:
    cols_show = ["var_idx", "from_node_name", "to_node_name"] + cols_show
print(f"\nTop {topN} variables by |relative difference|:")
print(cmp_sorted.loc[:topN-1, cols_show].to_string(index=False))

# Save detailed comparison
out_dir = "compare_posterior_vs_flow_mean"
os.makedirs(out_dir, exist_ok=True)
cmp.to_csv(os.path.join(out_dir, "posterior_vs_flow_mean.csv"), index=False)
print(f"\nSaved full comparison to {os.path.join(out_dir, 'posterior_vs_flow_mean.csv')}")

# Optional note: if many NaNs in flow_data_mean or large deviations:
# - Verify var_idx mapping (0-based vs 1-based Flow index).
# - Ensure flow_data.csv uses same flow ordering/naming as posterior.
# - If duplicates should be summed rather than averaged, replace 'mean' with 'sum' in the aggregation above.

Compared posterior vs flow_data mean on 80/241 variables.
- mean |abs_diff|: 1085
- median |abs_diff|: 449.5
- mean |rel_diff|: 53.88
- >100% relative deviation: 70.0%
- >0.1 absolute deviation: 100.0%

Top 20 variables by |relative difference|:
 var_idx  posterior_mean  flow_data_mean     abs_diff    rel_diff  dup_count
      42        1468.710        0.000000  1468.710000 1468.710000          1
      31         325.003        0.000000   325.003000  325.003000          1
      27         325.003        0.000000   325.003000  325.003000          1
      87       -4298.401       18.000000 -4316.401000 -239.800056          1
      65         460.122        1.923057   458.198943  238.265948          1
      67         460.700        2.614787   458.085213  175.190280          1
      69         460.766        2.662328   458.103672  172.068815          1
      57         897.821        6.000000   891.821000  148.636833          1
      71         437.700        3.846114   433.853886  112.80

In [8]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
from typing import Tuple, List, Optional
from numpy.linalg import solve, lstsq

# -------------------------
# Config
# -------------------------
FLOW_FILE = "flow_data.csv"
SCORE_FILE = "flow_data_score.csv"  # optional
OUT_BALANCE = "mass_balance_optimization.csv"
OUT_REC = "reconciled_optimization_values.csv"
OUT_CMP = "reconciliation_comparison.csv"

# -------------------------
# Data loading and preparation
# -------------------------
def load_and_prepare_data() -> Tuple[pd.DataFrame, Optional[pd.DataFrame], bool]:
    print("Loading data...")
    df = pd.read_csv(FLOW_FILE)
    print(f"Loaded {len(df)} flow records")

    # Coerce numeric columns
    for col in ["Value1", "Value2", "Value3", "Value4", "from_node_number", "to_node_number", "Flow index"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Load scores if available
    has_scores = False
    df_scores = None
    if os.path.exists(SCORE_FILE):
        df_scores = pd.read_csv(SCORE_FILE)
        print(f"Loaded scores data with {len(df_scores)} records")
        for col in ["Score1", "Score2", "Score3", "Score4", "Flow index"]:
            if col in df_scores.columns:
                df_scores[col] = pd.to_numeric(df_scores[col], errors="coerce")
        has_scores = True
    else:
        print("Warning: flow_data_score.csv not found. Using equal weights.")

    # Merge scores onto data if Flow index exists on both; otherwise keep separate
    if has_scores and ("Flow index" in df.columns) and ("Flow index" in df_scores.columns):
        df = df.merge(df_scores, on="Flow index", how="left", suffixes=("", "_score"))
        df_scores = None  # merged in
        has_scores = True

    # Keep original row for tracking
    df["original_idx"] = np.arange(len(df))

    return df, df_scores, has_scores

# -------------------------
# Weighted mean and uncertainty
# -------------------------
def calculate_weighted_means_and_uncertainties(
    df: pd.DataFrame, df_scores: Optional[pd.DataFrame], has_scores: bool
) -> Tuple[np.ndarray, np.ndarray]:
    print("Calculating weighted means and uncertainties...")
    value_cols = [c for c in ["Value1", "Value2", "Value3", "Value4"] if c in df.columns]
    score_cols = [c for c in ["Score1", "Score2", "Score3", "Score4"] if c in (df.columns if df_scores is None else df_scores.columns)]

    weighted_means: List[float] = []
    uncertainties: List[float] = []

    for i in range(len(df)):
        # Values
        vals = []
        for c in value_cols:
            v = df.at[i, c]
            if pd.notna(v):
                vals.append(float(v))
        vals = np.array(vals, dtype=float)

        # Scores aligned with values count
        if has_scores:
            if df_scores is None:
                # scores came via merge onto df
                scs = []
                for c in score_cols[: len(vals)]:
                    sv = df.at[i, c] if c in df.columns else np.nan
                    if pd.notna(sv):
                        scs.append(float(sv))
                scs = np.array(scs, dtype=float)
            else:
                scs = []
                for c in score_cols[: len(vals)]:
                    sv = df_scores.at[i, c] if c in df_scores.columns else np.nan
                    if pd.notna(sv):
                        scs.append(float(sv))
                scs = np.array(scs, dtype=float)
        else:
            scs = np.ones(len(vals), dtype=float)

        if vals.size == 0:
            # No valid values — set default mean/uncertainty
            wm = 100.0
            unc = 50.0
        else:
            if scs.size != vals.size or scs.size == 0 or float(np.nansum(scs)) <= 0:
                scs = np.ones_like(vals)
            w = scs / np.sum(scs)
            wm = float(np.sum(vals * w))

            if vals.size > 1:
                base_unc = float(np.std(vals))  # spread among values
                avg_score = float(np.mean(scs))
                score_factor = 2.0 - min(avg_score, 1.0)  # 1.0..2.0
                unc = max(base_unc * score_factor, 0.01 * abs(wm))
            else:
                avg_score = float(np.mean(scs))
                unc = max(0.1 * abs(wm) * (2.0 - min(avg_score, 1.0)), 1.0)

        weighted_means.append(wm)
        uncertainties.append(unc)

    return np.array(weighted_means, dtype=float), np.array(uncertainties, dtype=float)

# -------------------------
# Mass-balance constraints
# -------------------------
def build_mass_balance_constraints(df: pd.DataFrame) -> Tuple[np.ndarray, List[int]]:
    print("Building mass balance constraints...")
    all_from = df["from_node_number"].dropna().unique() if "from_node_number" in df.columns else np.array([])
    all_to = df["to_node_number"].dropna().unique() if "to_node_number" in df.columns else np.array([])
    nodes = sorted(set(all_from) | set(all_to))
    # Internal nodes: appear in both columns
    internal = []
    for n in nodes:
        has_in = bool(np.any(df["to_node_number"] == n)) if "to_node_number" in df.columns else False
        has_out = bool(np.any(df["from_node_number"] == n)) if "from_node_number" in df.columns else False
        if has_in and has_out:
            internal.append(n)

    print(f"Intermediate nodes (with mass balance): {len(internal)}")

    n_flows = len(df)
    A_rows = []
    for node in internal:
        row = np.zeros(n_flows, dtype=float)
        # inflows positive
        infl_idx = np.where(df["to_node_number"] == node)[0] if "to_node_number" in df.columns else []
        # outflows negative
        out_idx = np.where(df["from_node_number"] == node)[0] if "from_node_number" in df.columns else []
        row[infl_idx] += 1.0
        row[out_idx] -= 1.0
        if np.any(row != 0.0):
            A_rows.append(row)

    A = np.vstack(A_rows) if A_rows else np.zeros((0, n_flows), dtype=float)
    print(f"Active mass balance constraints: {A.shape[0]}; sparsity={np.count_nonzero(A)/max(1,A.size):.3f}")
    return A, internal

# -------------------------
# Fast KKT solve with active-set for x >= 0
# -------------------------
def reconcile_fast(weighted_means: np.ndarray, uncertainties: np.ndarray, A: np.ndarray, max_iter: int = 50) -> np.ndarray:
    """
    Solve min 0.5 (x - m)^T W (x - m) s.t. A x = 0, x >= 0
    W = diag(1/unc^2). Active-set ensures non-negativity.
    """
    m = weighted_means
    w = 1.0 / np.maximum(uncertainties, 1e-12) ** 2
    n = m.size
    A = np.asarray(A, dtype=float)

    free = np.ones(n, dtype=bool)  # all variables free initially
    x = np.maximum(m, 0.0)

    for it in range(max_iter):
        # Build reduced system on free variables
        idx = np.where(free)[0]
        if idx.size == 0:
            break

        Wf = np.diag(w[idx])
        Af = A[:, idx] if A.size > 0 else np.zeros((0, idx.size))
        mf = m[idx]

        # KKT: [Wf Af^T; Af 0] [x_f; lam] = [Wf mf; 0]
        # Solve robustly with small jitter
        if Af.size == 0:
            xf = mf.copy()
        else:
            K11 = Wf
            K12 = Af.T
            K21 = Af
            K22 = np.zeros((Af.shape[0], Af.shape[0]))
            KKT = np.block([[K11, K12], [K21, K22]])
            rhs = np.concatenate([Wf @ mf, np.zeros(Af.shape[0])])

            try:
                sol = solve(KKT + 1e-10 * np.eye(KKT.shape[0]), rhs, assume_a='sym')
            except Exception:
                sol = lstsq(KKT + 1e-10 * np.eye(KKT.shape[0]), rhs, rcond=None)[0]
            xf = sol[: idx.size]

        # Assemble full x, apply non-negativity
        x_new = np.zeros_like(x)
        x_new[idx] = xf
        x_new[~free] = 0.0

        # Identify negatives to fix at 0
        neg = (x_new < 0.0) & free
        if not np.any(neg):
            x = x_new
            break
        # Fix new negatives
        free[neg] = False
        x = x_new

    # Final projection to exact mass balance (should already hold if KKT solved)
    if A.size > 0:
        rhs = A @ x
        if np.linalg.norm(rhs) > 1e-10:
            AAT = A @ A.T
            try:
                lam = solve(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs, assume_a='pos')
            except Exception:
                lam = lstsq(AAT + 1e-10 * np.eye(AAT.shape[0]), rhs, rcond=None)[0]
            x = x - A.T @ lam

    # Ensure non-negativity after projection (rarely needed); if violated, re-run one more set/fix
    x[x < 0.0] = 0.0
    return x

# -------------------------
# Verification and save
# -------------------------
def verify_mass_balance(df: pd.DataFrame, x: np.ndarray, internal_nodes: List[int]) -> float:
    print("\n=== Mass Balance Verification ===")
    n = len(df)
    recs = []
    max_abs = 0.0
    for node in internal_nodes:
        infl_idx = [j for j in range(n) if df.iloc[j]["to_node_number"] == node]
        out_idx = [j for j in range(n) if df.iloc[j]["from_node_number"] == node]
        infl = x[infl_idx].sum() if infl_idx else 0.0
        out = x[out_idx].sum() if out_idx else 0.0
        imb = infl - out
        tot = max(infl, out, 1e-6)
        rel = 100.0 * abs(imb) / tot
        max_abs = max(max_abs, abs(imb))
        recs.append({"node_number": node, "total_inflow": infl, "total_outflow": out, "imbalance": imb, "imbalance_percent": rel})
    if recs:
        dfmb = pd.DataFrame(recs).sort_values("imbalance_percent", ascending=False)
        print("Worst 5 mass balance violations:")
        print(dfmb.head()[["node_number", "total_inflow", "total_outflow", "imbalance", "imbalance_percent"]])
        dfmb.to_csv(OUT_BALANCE, index=False)
        print(f"Mass balance verification saved to: {OUT_BALANCE}")
    print(f"Maximum absolute imbalance: {max_abs:.6e}")
    return max_abs

def save_results(df: pd.DataFrame, x: np.ndarray, m: np.ndarray, s: np.ndarray):
    print("\n=== Saving Results ===")
    results = df.copy()
    results["original_mean"] = m
    results["uncertainty"] = s
    results["reconciled_value"] = x
    results["adjustment"] = x - m
    results["relative_adjustment"] = 100.0 * results["adjustment"] / np.where(m != 0.0, m, 1.0)
    results["weighted_residual"] = results["adjustment"] / np.where(s > 0, s, 1.0)

    # Main results
    out_cols = [
        "Flow index", "from_node_name", "to_node_name", "from_node_number", "to_node_number",
        "original_mean", "uncertainty", "reconciled_value", "adjustment", "relative_adjustment"
    ]
    keep_cols = [c for c in out_cols if c in results.columns]
    results[keep_cols].to_csv(OUT_REC, index=False)
    print(f"Reconciled values saved to: {OUT_REC}")

    # Summary
    print("\n=== Adjustment Analysis ===")
    print(f"Mean absolute adjustment: {results['adjustment'].abs().mean():.3g}")
    print(f"Mean relative adjustment: {results['relative_adjustment'].abs().mean():.3g}%")
    print(f"Max absolute adjustment: {results['adjustment'].abs().max():.3g}")
    print(f"Max relative adjustment: {results['relative_adjustment'].abs().max():.3g}%")
    print(f"RMS weighted residual: {np.sqrt((results['weighted_residual']**2).mean()):.3g}")

    top = results.nlargest(10, "relative_adjustment", keep="all")[[
        c for c in ["from_node_name", "to_node_name", "original_mean", "reconciled_value", "relative_adjustment"] if c in results.columns
    ]]
    print("\nLargest relative adjustments:")
    print(top)

    # Detailed comparison
    cmp_cols = [c for c in [
        "from_node_name", "to_node_name", "original_mean", "reconciled_value", "adjustment", "relative_adjustment", "weighted_residual"
    ] if c in results.columns]
    results[cmp_cols].to_csv(OUT_CMP, index=False)
    print(f"Detailed comparison saved to: {OUT_CMP}")

# -------------------------
# Main
# -------------------------
def main():
    print("=== Fast Optimization-Based Mass Flow Reconciliation (KKT + Active-Set) ===")
    df, df_scores, has_scores = load_and_prepare_data()
    m, s = calculate_weighted_means_and_uncertainties(df, df_scores, has_scores)
    A, internal_nodes = build_mass_balance_constraints(df)

    print("\n=== Solving (fast KKT) ===")
    t0 = time.time()
    x = reconcile_fast(m, s, A, max_iter=50)
    t1 = time.time()
    print(f"Solved in {t1 - t0:.3f} s")

    max_imbalance = verify_mass_balance(df, x, internal_nodes)
    save_results(df, x, m, s)

    print("\n=== Summary ===")
    print(f"Variables: {len(x)}, constraints: {A.shape[0]}")
    print(f"Max mass balance violation: {max_imbalance:.3e}")
    if max_imbalance < 1e-6:
        print("[SUCCESS] Mass balance satisfied to machine precision")
    elif max_imbalance < 1e-3:
        print("[OK] Mass balance satisfied to reasonable tolerance")
    else:
        print("[WARNING] Mass balance may not be fully satisfied")

if __name__ == "__main__":
    main()

=== Fast Optimization-Based Mass Flow Reconciliation (KKT + Active-Set) ===
Loading data...
Loaded 241 flow records
Loaded scores data with 241 records
Calculating weighted means and uncertainties...
Building mass balance constraints...
Intermediate nodes (with mass balance): 62
Active mass balance constraints: 62; sparsity=0.027

=== Solving (fast KKT) ===
Solved in 0.000 s

=== Mass Balance Verification ===
Worst 5 mass balance violations:
    node_number  total_inflow  total_outflow     imbalance  imbalance_percent
28           41      5.000000       5.000000  4.454215e-11       8.908430e-10
15           28      5.939787       5.939787 -1.003997e-11       1.690291e-10
27           40     57.653329      57.653329  4.654765e-11       8.073715e-11
17           30      7.961764       7.961764 -5.741185e-12       7.210947e-11
53           67     63.215986      63.215986 -2.487610e-11       3.935097e-11
Mass balance verification saved to: mass_balance_optimization.csv
Maximum absolute imb

In [10]:
import os
import numpy as np
import pandas as pd
from numpy.linalg import solve, lstsq

FLOW_FILE = "flow_data.csv"
SCORE_FILE = "flow_data_score.csv"  # optional

TARGET_N = 241
CHECK_IDXS = [236, 237, 238, 239, 240]

def load_data():
    df = pd.read_csv(FLOW_FILE)
    # Coerce numeric
    for col in ["Flow index", "from_node_number", "to_node_number", "Value1", "Value2", "Value3", "Value4"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    # Optional scores
    df_score = None
    if os.path.exists(SCORE_FILE):
        df_score = pd.read_csv(SCORE_FILE)
        for col in ["Flow index", "Score1", "Score2", "Score3", "Score4"]:
            if col in df_score.columns:
                df_score[col] = pd.to_numeric(df_score[col], errors="coerce")
    return df, df_score

def align_by_flow_index(df):
    if "Flow index" not in df.columns:
        raise ValueError("flow_data.csv must contain 'Flow index'")
    # Auto detect base
    fi_nonan = df["Flow index"].dropna().astype(int)
    zero_based = (fi_nonan.min() == 0)
    offset = 0 if zero_based else 1

    df = df.dropna(subset=["Flow index"]).copy()
    df["var_idx"] = (df["Flow index"].astype(int) - offset)
    # Keep only 0..240 and deduplicate per var_idx (median Value1)
    df = df.loc[(df["var_idx"] >= 0) & (df["var_idx"] < TARGET_N)].copy()

    # Aggregate values per var_idx
    val_cols = [c for c in ["Value1", "Value2", "Value3", "Value4"] if c in df.columns]
    agg = (df.groupby("var_idx", as_index=False)
             .agg(**{c: (c, "median") for c in val_cols},
                  from_node_number=("from_node_number", "first"),
                  to_node_number=("to_node_number", "first")))
    # Ensure all 0..240 are present
    full = pd.DataFrame({"var_idx": np.arange(TARGET_N)})
    agg = full.merge(agg, on="var_idx", how="left")
    return agg

def compute_means_and_uncertainty(agg, df_score):
    # Weighted mean across Value1..4 using Score1..4 if present; else Value1 only
    val_cols = [c for c in ["Value1", "Value2", "Value3", "Value4"] if c in agg.columns]
    # Build weights from score file if it aligns by Flow index; else equal weights
    weights = None
    if df_score is not None and "Flow index" in df_score.columns:
        # Map scores to var_idx
        fi_nonan = df_score["Flow index"].dropna().astype(int)
        zero_based = (fi_nonan.min() == 0)
        offset = 0 if zero_based else 1
        df_score = df_score.dropna(subset=["Flow index"]).copy()
        df_score["var_idx"] = df_score["Flow index"].astype(int) - offset
        df_score = df_score.loc[(df_score["var_idx"] >= 0) & (df_score["var_idx"] < TARGET_N)]
        sc_cols = [c for c in ["Score1", "Score2", "Score3", "Score4"] if c in df_score.columns]
        if sc_cols:
            sc = df_score.set_index("var_idx")[sc_cols].reindex(np.arange(TARGET_N))
            weights = sc.to_numpy(dtype=float)
    # Collect values
    V = agg[val_cols].to_numpy(dtype=float)
    # Compute weighted means
    m = np.zeros(TARGET_N, dtype=float)
    s = np.zeros(TARGET_N, dtype=float)
    for j in range(TARGET_N):
        vals = V[j, :] if V.shape[1] > 0 else np.array([np.nan])
        vals = vals[~np.isnan(vals)]
        if vals.size == 0:
            m[j] = 0.0
            s[j] = 1e6  # essentially uninformative
            continue
        if weights is not None:
            wj = weights[j, :vals.size]
            wj = np.where(np.isfinite(wj), wj, 0.0)
            if wj.sum() <= 0:
                wj = np.ones_like(vals)
        else:
            # Prefer Value1 if present
            if "Value1" in val_cols and not np.isnan(agg.loc[j, "Value1"]):
                vals = np.array([agg.loc[j, "Value1"]], dtype=float)
                wj = np.array([1.0], dtype=float)
            else:
                wj = np.ones_like(vals)
        wj = wj / wj.sum()
        m[j] = float(np.sum(vals * wj))
        # uncertainty: 10% of mean as default, or spread if multiple entries
        base = np.std(vals) if vals.size > 1 else max(0.1 * abs(m[j]), 1.0)
        s[j] = max(base, 0.01 * abs(m[j]))
    return m, s

def build_mass_balance(agg):
    n = TARGET_N
    A_rows = []
    # Internal nodes = appear in both from/to columns anywhere
    from_all = agg["from_node_number"].dropna().unique()
    to_all = agg["to_node_number"].dropna().unique()
    internal_nodes = set(from_all).intersection(set(to_all))
    for node in sorted(internal_nodes):
        row = np.zeros(n, dtype=float)
        infl = agg.index[agg["to_node_number"] == node].to_numpy()
        outf = agg.index[agg["from_node_number"] == node].to_numpy()
        if infl.size > 0 and outf.size > 0:
            row[infl] += 1.0
            row[outf] -= 1.0
            A_rows.append(row)
    A = np.vstack(A_rows) if A_rows else np.zeros((0, n), dtype=float)
    return A

def solve_kkt_with_bounds(m, s, A, lb, ub, max_iter=50, tol=1e-10):
    """
    Solve min 0.5 (x-m)^T W (x-m) s.t. A x = 0, lb <= x <= ub
    via active-set on bounds; equality handled by KKT each iteration.
    """
    n = m.size
    W = 1.0 / np.maximum(s, 1e-12) ** 2
    x = np.clip(m, lb, ub)
    fixed = np.zeros(n, dtype=bool)  # indices fixed at bounds

    for _ in range(max_iter):
        free = ~fixed
        if not free.any():
            break
        Af = A[:, free] if A.size else np.zeros((0, free.sum()))
        Ab = A[:, fixed] if A.size else np.zeros((0, fixed.sum()))
        Wf = np.diag(W[free])
        mf = m[free]
        xb = x[fixed]

        # KKT system for free vars: Wf x_f + Af^T λ = Wf m_f; Af x_f = -Ab x_b
        K11 = Wf
        K12 = Af.T
        K21 = Af
        K22 = np.zeros((Af.shape[0], Af.shape[0]))
        KKT = np.block([[K11, K12], [K21, K22]])
        rhs = np.concatenate([Wf @ mf, -Ab @ xb]) if A.size else np.concatenate([Wf @ mf, np.zeros(0)])

        try:
            sol = solve(KKT + 1e-10*np.eye(KKT.shape[0]), rhs, assume_a='sym')
        except Exception:
            sol = lstsq(KKT + 1e-10*np.eye(KKT.shape[0]), rhs, rcond=None)[0]

        xf = sol[:free.sum()]
        x_new = x.copy()
        x_new[free] = xf

        # Check bounds
        below = (x_new < lb - 1e-12)
        above = (x_new > ub + 1e-12)
        if not (below | above).any():
            x = np.clip(x_new, lb, ub)
            break
        # Fix newly violated indices to nearest bound
        newly_fix = (below | above) & (~fixed)
        x[newly_fix & below] = lb[newly_fix & below]
        x[newly_fix & above] = ub[newly_fix & above]
        fixed |= newly_fix

    # Final tiny projection to equality (should be near-zero already)
    if A.size:
        rhs_eq = A @ x
        if np.linalg.norm(rhs_eq) > tol:
            AAT = A @ A.T
            try:
                lam = solve(AAT + 1e-10*np.eye(AAT.shape[0]), rhs_eq, assume_a='pos')
            except Exception:
                lam = lstsq(AAT + 1e-10*np.eye(AAT.shape[0]), rhs_eq, rcond=None)[0]
            x = x - A.T @ lam
            x = np.clip(x, lb, ub)
    return x

# Run pipeline
df, df_score = load_data()
agg = align_by_flow_index(df)
m, s = compute_means_and_uncertainty(agg, df_score)

# Bounds: observed flows get [0.2*obs, 1.8*obs]; missing get [0, wide]
obs = agg["Value1"].to_numpy(dtype=float)  # primary observed
obs_mask = np.isfinite(obs)
max_obs = np.nanmax(obs) if np.any(obs_mask) else 1.0
wide_ub = max(10.0 * max_obs, 1e3)  # generous upper bound for missing

lb = np.where(obs_mask, 0.2 * obs, 0.0)
ub = np.where(obs_mask, 1.8 * obs, wide_ub)
# Make sure lb <= ub
lb = np.minimum(lb, ub - 1e-9)

A = build_mass_balance(agg)
x_rec = solve_kkt_with_bounds(m, s, A, lb, ub)

# Focused check for 236..240
print("\nCheck indices 236..240 (obs, lb, ub, mean, reconciled):")
for j in CHECK_IDXS:
    print(f"j={j:3d} | obs={obs[j]} | lb={lb[j]} | ub={ub[j]} | mean(m)={m[j]} | x={x_rec[j]}")

# Save results table
results = agg.copy()
results["original_mean"] = m
results["uncertainty"] = s
results["reconciled_value"] = x_rec
results["lower_bound"] = lb
results["upper_bound"] = ub
results.to_csv("reconciled_optimization_values.csv", index=False)
print("\nSaved reconciled_optimization_values.csv")

# Quick sanity summary
imb = A @ x_rec if A.size else np.array([])
if imb.size:
    print(f"Mass-balance residuals: min={imb.min():.3e}, max={imb.max():.3e}, L2={np.linalg.norm(imb):.3e}")


Check indices 236..240 (obs, lb, ub, mean, reconciled):
j=236 | obs=nan | lb=0.0 | ub=79580.0 | mean(m)=224.0 | x=222.19082109546693
j=237 | obs=nan | lb=0.0 | ub=79580.0 | mean(m)=287.0 | x=284.0300439892697
j=238 | obs=nan | lb=0.0 | ub=79580.0 | mean(m)=754.0 | x=733.5009219611502
j=239 | obs=nan | lb=0.0 | ub=79580.0 | mean(m)=1608.0 | x=1514.7662535981717
j=240 | obs=nan | lb=0.0 | ub=79580.0 | mean(m)=247.0 | x=244.80021793290567

Saved reconciled_optimization_values.csv
Mass-balance residuals: min=-1.876e-12, max=1.116e-11, L2=1.227e-11


In [11]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import arviz as az
from scipy.linalg import cholesky, solve_triangular, eigh
from numpy.linalg import lstsq

# Configuration
FLOW_FILE = "flow_data.csv"
SCORE_FILE = "flow_data_score.csv"   # optional, not required here
RECONCILED_CSV = "reconciled_optimization_values.csv"  # produced by your optimizer
OUT_DIR = "full_space_res_samples"
OUT_PREFIX = "full_space_res_2022"
N_CHAINS = 4
N_DRAWS = 2000   # total samples = N_CHAINS * N_DRAWS
SEED = 42
CLIP_ITER = 2    # alternate clipping+projection iterations

np.random.seed(SEED)

# Helper: parse reconciled results or recompute minimal aggregates if missing
def load_reconciled_or_aggregate(flow_file=FLOW_FILE, reconciled_csv=RECONCILED_CSV, target_n=241):
    df = pd.read_csv(flow_file)
    # Ensure numeric
    for c in ["Flow index", "from_node_number", "to_node_number", "Value1", "Value2", "Value3", "Value4"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # If reconciled CSV exists, load reconciled values and bounds from it
    if os.path.exists(reconciled_csv):
        rec = pd.read_csv(reconciled_csv)
        # Expect reconciled_value column; if var order not explicit, we align by Flow index
        if "reconciled_value" not in rec.columns:
            raise RuntimeError(f"{reconciled_csv} missing 'reconciled_value' column")
        # Try to align via Flow index if present; otherwise assume same order
        if "Flow index" in rec.columns:
            rec = rec.sort_values("Flow index").reset_index(drop=True)
        return df, rec

    # Otherwise build aggregated table (median per var index) like earlier pipeline
    if "Flow index" not in df.columns:
        raise RuntimeError("flow_data.csv must include 'Flow index' when reconciled csv not present.")
    fi = df["Flow index"].dropna().astype(int)
    zero_based = fi.min() == 0
    offset = 0 if zero_based else 1
    df = df.dropna(subset=["Flow index"]).copy()
    df["var_idx"] = df["Flow index"].astype(int) - offset
    # limit to first target_n
    df = df[(df["var_idx"] >= 0) & (df["var_idx"] < target_n)]
    agg = (
        df.groupby("var_idx", as_index=False)
          .agg(Value1=("Value1", "median"),
               from_node_number=("from_node_number", "first"),
               to_node_number=("to_node_number", "first"))
    )
    # fill missing indices
    full = pd.DataFrame({"var_idx": np.arange(target_n)})
    agg = full.merge(agg, on="var_idx", how="left")
    # create "reconciled" placeholder equal to median (or NaN)
    agg["reconciled_value"] = agg["Value1"].fillna(0.0)
    return df, agg

# Build mass-balance constraint matrix A given aggregated table with from/to node numbers
def build_A_from_agg(agg):
    n = agg.shape[0]
    from_all = agg["from_node_number"].dropna().unique() if "from_node_number" in agg.columns else np.array([])
    to_all = agg["to_node_number"].dropna().unique() if "to_node_number" in agg.columns else np.array([])
    internal_nodes = set(from_all).intersection(set(to_all))
    A_rows = []
    nodes = []
    for node in sorted(internal_nodes):
        infl = agg.index[agg["to_node_number"] == node].to_numpy()
        outf = agg.index[agg["from_node_number"] == node].to_numpy()
        if infl.size > 0 and outf.size > 0:
            row = np.zeros(n, dtype=float)
            row[infl] += 1.0
            row[outf] -= 1.0
            A_rows.append(row)
            nodes.append(node)
    A = np.vstack(A_rows) if A_rows else np.zeros((0, n), dtype=float)
    return A, nodes

# Projection to mass-balance for a batch of samples X (n_vars x n_samples)
def project_to_mass_balance_batch(A, X):
    if A.size == 0:
        return X
    AAT = A @ A.T
    # factor AAT once (try Cholesky)
    try:
        L = cholesky(AAT + 1e-12 * np.eye(AAT.shape[0]), lower=True)
        # solve L y = A X  => y = L^{-1} (A X)
        Y = np.linalg.solve(L, A @ X)
        # solve L^T lam = Y
        lam = np.linalg.solve(L.T, Y)
    except Exception:
        # fallback to solve per RHS
        lam = lstsq(AAT + 1e-12 * np.eye(AAT.shape[0]), A @ X, rcond=None)[0]
    X_proj = X - A.T @ lam
    return X_proj

# Main sampling routine using Laplace approximation in nullspace
def sample_posterior_from_reconciled(df, rec, n_chains=N_CHAINS, n_draws=N_DRAWS):
    # Build aggregated table 'agg' where each row corresponds to variable index 0..n-1
    if "var_idx" in rec.columns:
        agg = rec.copy()
        agg = agg.sort_values("var_idx").reset_index(drop=True)
    else:
        # If rec was raw df aggregation
        agg = rec.copy()
        if "var_idx" in agg.columns:
            agg = agg.sort_values("var_idx").reset_index(drop=True)
    n_vars = agg.shape[0]

    # Observations and estimated uncertainties: try to infer from rec if present
    # Use 'original_mean' or 'Value1' as observation, and 'uncertainty' column if provided
    if "original_mean" in agg.columns:
        m = agg["original_mean"].to_numpy(dtype=float)
    elif "Value1" in agg.columns:
        m = agg["Value1"].fillna(0.0).to_numpy(dtype=float)
    else:
        m = agg.get("reconciled_value", np.zeros(n_vars)).to_numpy(dtype=float)

    # For sampling we need an estimate of noise/uncertainty per variable.
    # Prefer 'uncertainty' column if present; else use heuristic 10% of m with floor
    if "uncertainty" in agg.columns:
        s = agg["uncertainty"].to_numpy(dtype=float)
        # replace 0 or nan with heuristic
        s = np.where(np.isfinite(s) & (s > 1e-12), s, np.maximum(0.10 * np.abs(m), 1e-3))
    else:
        s = np.maximum(0.10 * np.abs(m), 1e-3)

    # Bounds if present in rec; else use wide bounds (clip after sampling)
    lb = agg["lower_bound"].to_numpy(dtype=float) if "lower_bound" in agg.columns else np.zeros(n_vars)
    ub = agg["upper_bound"].to_numpy(dtype=float) if "upper_bound" in agg.columns else np.full(n_vars, np.max(np.abs(m)) * 10 + 1e3)
    lb = np.where(np.isnan(lb), 0.0, lb)
    ub = np.where(np.isnan(ub), np.max(np.abs(m)) * 10 + 1e3, ub)

    # Use reconciled_value as MAP (if present) else m
    x_map = agg["reconciled_value"].to_numpy(dtype=float) if "reconciled_value" in agg.columns else m.copy()

    # Build mass-balance matrix A from aggregated table (requires from_node_number & to_node_number)
    A, internal_nodes = build_A_from_agg(agg)

    # Build nullspace Z of A: columns span null(A)
    if A.size == 0:
        Z = np.eye(n_vars)
    else:
        # SVD for nullspace
        U, svals, Vt = np.linalg.svd(A, full_matrices=True)
        rank = (svals > (1e-10 * svals.max())).sum()
        Z = Vt[rank:].T    # n_vars x k

    k = Z.shape[1]
    print(f"Sampling in nullspace of dimension k = {k} (n_vars = {n_vars}); A rows = {A.shape[0]}")

    # Precision Q = diag(1/s^2) (weights from uncertainties)
    Q = np.diag(1.0 / np.maximum(s, 1e-12) ** 2)

    # Effective precision in nullspace: Q_eff = Z^T Q Z
    Q_eff = Z.T @ Q @ Z
    # regularize
    Q_eff += 1e-12 * np.eye(k)

    # Factorize Q_eff. We'll draw Z0 ~ N(0, I_k x S) and obtain g = L^{-1} Z0 where L = cholesky(Q_eff)
    use_chol = True
    try:
        L_eff = cholesky(Q_eff, lower=True)
    except Exception:
        use_chol = False
        # fallback eigen decomposition
        w, V = eigh(Q_eff)
        w = np.maximum(w, 1e-12)
        L_eff = V @ np.diag(np.sqrt(w))  # not triangular

    total_samps = n_chains * n_draws
    rng = np.random.default_rng(SEED)

    # Draw Z0 of shape (k, total_samps)
    if k == 0:
        # No degrees of freedom (all variables constrained), samples are all equal to x_map
        X = np.tile(x_map[:, None], (1, total_samps))
    else:
        Z0 = rng.standard_normal(size=(k, total_samps))
        if use_chol:
            # solve L_eff g = Z0  -> g = L_eff^{-1} Z0, so covariance = inv(Q_eff)
            g = solve_triangular(L_eff, Z0, lower=True)   # k x S
        else:
            # solve L_eff g = Z0 using lstsq (L_eff not triangular)
            g = lstsq(L_eff, Z0, rcond=None)[0]

        # Map to full space: X = x_map[:,None] + Z @ g
        X = x_map[:, None] + (Z @ g)  # shape n_vars x total_samps

    # Optionally clip to bounds and re-project to mass balance a few times
    for it in range(CLIP_ITER):
        X = np.clip(X, lb[:, None], ub[:, None])
        if A.size:
            X = project_to_mass_balance_batch(A, X)

    # Now build samples array shaped (n_chains, n_draws, n_vars)
    S = total_samps
    if S != N_CHAINS * N_DRAWS:
        # fallback: reshape by truncation/padding
        S = N_CHAINS * N_DRAWS
        if X.shape[1] >= S:
            X = X[:, :S]
        else:
            # pad by repeating last column
            pad = np.tile(X[:, -1][:, None], (1, S - X.shape[1]))
            X = np.hstack([X, pad])

    samples = X.T.reshape(N_CHAINS, N_DRAWS, n_vars)

    # Build arviz InferenceData
    coords = {"chain": np.arange(N_CHAINS), "draw": np.arange(N_DRAWS), "mu_x_dim_0": np.arange(n_vars)}
    posterior_ds = xr.Dataset({"mu_x": (("chain", "draw", "mu_x_dim_0"), samples)}, coords=coords)
    idata = az.InferenceData(posterior=posterior_ds)

    # Compute posterior mean/cov and diagnostics
    flat = samples.reshape(-1, n_vars)  # S x n_vars
    post_mean = flat.mean(axis=0)
    post_cov = np.cov(flat, rowvar=False)
    # Mass-balance residuals for posterior mean and for samples
    if A.size:
        resid_mean = A @ post_mean
        resid_per_sample = A @ flat.T   # m x S
        max_abs_resid = np.max(np.abs(resid_per_sample))
    else:
        resid_mean = np.array([])
        max_abs_resid = 0.0

    diagnostics = {
        "n_vars": n_vars,
        "n_constraints": A.shape[0],
        "nullspace_dim": k,
        "total_samples": total_samps,
        "max_abs_mass_balance_residual": float(max_abs_resid),
        "resid_mean_norm": float(np.linalg.norm(resid_mean)) if resid_mean.size else 0.0,
    }

    return idata, post_mean, post_cov, diagnostics, x_map, lb, ub, agg

# Run
df, rec = load_reconciled_or_aggregate()
idata, post_mean, post_cov, diagnostics, x_map, lb, ub, agg = sample_posterior_from_reconciled(df, rec, n_chains=N_CHAINS, n_draws=N_DRAWS)

# Print diagnostics
print("Sampling diagnostics:")
for k, v in diagnostics.items():
    print(f"  {k}: {v}")

# Focused check for variables 236..240 if present
check_idxs = [236, 237, 238, 239, 240]
n_vars = post_mean.shape[0]
print("\nCheck indices 236..240 (obs, lb, ub, map, posterior_mean):")
for j in check_idxs:
    if j < n_vars:
        obs = agg.loc[j, "Value1"] if "Value1" in agg.columns else np.nan
        print(f"j={j} | obs={obs} | lb={lb[j]:.6g} | ub={ub[j]:.6g} | map={x_map[j]:.6g} | post_mean={post_mean[j]:.6g}")
    else:
        print(f"j={j} not in range (n_vars={n_vars})")

# Mass-balance check on samples (compute A @ sample for a few samples)
if "posterior" in idata:
    S_flat = idata.posterior["mu_x"].stack(sample=("chain", "draw")).transpose("mu_x_dim_0", "sample").values
    if diagnostics["n_constraints"] > 0:
        A, _ = build_A_from_agg(agg)
        resid_samples = A @ S_flat  # m x S
        print("\nMass-balance residuals across samples summary (per constraint):")
        print("  max abs:", np.max(np.abs(resid_samples)))
        print("  mean abs:", np.mean(np.abs(resid_samples)))
        print("  std:", np.std(resid_samples))

# Save outputs
os.makedirs(OUT_DIR, exist_ok=True)
az.to_netcdf(idata, os.path.join(OUT_DIR, f"{OUT_PREFIX}.nc"))
np.save(os.path.join(OUT_DIR, f"{OUT_PREFIX}.npy"), idata.posterior["mu_x"].stack(sample=("chain", "draw")).transpose("mu_x_dim_0", "sample").values)
pd.DataFrame({"posterior_mean": post_mean}).to_csv(os.path.join(OUT_DIR, f"{OUT_PREFIX}_posterior_mean.csv"), index=False)
# Save posterior covariance (may be large)
np.save(os.path.join(OUT_DIR, f"{OUT_PREFIX}_posterior_cov.npy"), post_cov)

print("\nSaved posterior samples and summaries to", OUT_DIR)

Sampling in nullspace of dimension k = 179 (n_vars = 241); A rows = 62
Sampling diagnostics:
  n_vars: 241
  n_constraints: 62
  nullspace_dim: 179
  total_samples: 8000
  max_abs_mass_balance_residual: 2.5920599000528455e-11
  resid_mean_norm: 8.297145990338322e-12

Check indices 236..240 (obs, lb, ub, map, posterior_mean):
j=236 | obs=nan | lb=0 | ub=79580 | map=222.191 | post_mean=222.34
j=237 | obs=nan | lb=0 | ub=79580 | map=284.03 | post_mean=283.817
j=238 | obs=nan | lb=0 | ub=79580 | map=733.501 | post_mean=733.253
j=239 | obs=nan | lb=0 | ub=79580 | map=1514.77 | post_mean=1513.18
j=240 | obs=nan | lb=0 | ub=79580 | map=244.8 | post_mean=243.289

Mass-balance residuals across samples summary (per constraint):
  max abs: 2.5920599000528455e-11
  mean abs: 1.2371553134813711e-12
  std: 1.9155600353573655e-12

Saved posterior samples and summaries to full_space_res_samples
